# Task 2.2.4: Training our own YOLO Object Detection Model
## Setting up the Environment and Dataset Preparation

#### Environment Setup

In [ ]:
# Create a Virtual Environment
# Why? This will create an isolated environment where you can install the required packages without affecting your global Python environment.

# Run the following command in bash:
'''
python -m venv yolo_env
'''

In [ ]:
# Activate the Virtual Environment
# Why? Activating the environment will ensure that the packages you install and the code you run utilizes this isolated environment.

# Run the following command in bash:
'''
.\yolo_env\Scripts\activate
'''

In [ ]:
# Install Necessary Packages
# Why? These packages include libraries and tools that your project will depend on.

# Run the following command in bash:
'''
pip install torch torchvision torchaudio
'''

# If you do not have pip installed you can do so by running the following command in bash:
'''
python -m ensurepip --default-pip
'''

#### Cloning the YOLOv5 repository

In [ ]:
# Clone the YOLOv5 Repository
# Navigate to the folder where you want to clone the YOLOv5 repository, you can do so using the following bash commands:
'''
cd ..                   # To navigate up
cd [directory name]     # To navigate down
'''

# Run the following command in bash:
'''
git clone https://github.com/ultralytics/yolov5.git
'''

In [ ]:
# Navigate to the YOLOv5 Directory
# Why? You'll be running commands from inside this directory and using its codebase.

# Run the following command in bash:
'''
cd yolov5
'''

#### Installing Required Dependencies

In [ ]:
# Install required dependencies

# Run the following command in bash:
'''
pip install -U -r requirements.txt
'''

In [ ]:
# Install Jupyter Notebook Inside Your Virtual Environment
# Why? Future tasks will be using Python notebooks such as this one and the tasks beforehand, this will allow you to continue working on your YOLOv5 model from within these Python notebooks.

# Run the following command in bash:
'''
pip install notebook
'''

#### Preparing the Dataset

In [5]:
%ls

CV_LM2TU02_A10_T10_students.ipynb
CV_LM2TU02_A1_T1_students.ipynb
CV_LM2TU02_A2_T2_students.ipynb
CV_LM2TU02_A3_T3_students.ipynb
CV_LM2TU02_A4_T4_students.ipynb
CV_LM2TU02_A5_T5_students.ipynb
CV_LM2TU02_A6_T6_students.ipynb
CV_LM2TU02_A7_T7_students.ipynb
CV_LM2TU02_A8_T8_students.ipynb
CV_LM2TU02_A9_T9_students.ipynb
data/
data 2/
datasubset/
datasubset 2/
faster_rcnn_inception_v2_coco_2018_01_28/
faster_rcnn_inception_v2_coco_2018_01_28.tar.gz
imageset/
imageset 2/
kaggle/
models/
mscoco_label_map.pbtxt
preprocessedimageset/
preprocessedimageset 2/
test/
test 2/
train/
validate/
validate 2/
yolo/
yolov5s.pt


In [7]:
import pandas as pd
import os
from PIL import Image

# Load the CSV file
df = pd.read_csv('/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/kaggle/train_solution_bounding_boxes (1).csv')

# Function to convert bounding boxes to YOLO format
def convert_to_yolo_format(row):
    """
    Convert bounding box coordinates to YOLO format.
    """
    img = Image.open(os.path.join('/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/kaggle/training_images', row.image))
    width, height = img.size

    x_center = ((row.xmin + row.xmax) / 2) / width
    y_center = ((row.ymin + row.ymax) / 2) / height
    w = (row.xmax - row.xmin) / width
    h = (row.ymax - row.ymin) / height

    class_index = 0  # As there is only one class (car), the index is set to 0

    return f"{class_index} {x_center} {y_center} {w} {h}"

# Applying the function to each row in dataframe
df['yolo_format'] = df.apply(convert_to_yolo_format, axis=1)

# Save YOLO formatted strings to text files
for filename, group in df.groupby('image'):
    with open(f"/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/kaggle/training_annotations/{filename.split('.')[0]}.txt", 'w') as f:
        for annotation in group['yolo_format']:
            f.write(f"{annotation}\n")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/Weeks 4 & 5/kaggle/training_annotations/vid_4_1000.txt'

In [ ]:
# Some images in our training set do not contain a car, hence there is no label and bounding box available for it.
# However, the YOLO architecture requires each image to have a corresponding label `.txt` file, so we must create blank files for the remaining images.

import os

# Define the path to the directory where your images are stored
image_directory = "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/data/training_images"

# Define the path to the directory where your label files are stored
label_directory = "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/data/training_annotations"

# Get the list of all image filenames
image_filenames = os.listdir(image_directory)

# Go through each image filename
for image_filename in image_filenames:
    # Get the image name without the extension
    image_name, _ = os.path.splitext(image_filename)
    
    # Define the path to the corresponding label file
    label_filepath = os.path.join(label_directory, f"{image_name}.txt")
    
    # Check if the label file already exists
    if not os.path.exists(label_filepath):
        # If it doesn't exist, create an empty label file
        with open(label_filepath, 'w') as file:
            pass  # Empty file is created

    # print(f"Processed: {image_filename}")

print("Processing complete. Empty label files have been created where necessary.")


In [ ]:
'''
We have now created all the necessary label files for our data. Copy all the .txt files and put them into the same directory as the training images.
'''

#### Splitting dataset into training, validation and test images

In [ ]:
import os
import shutil
import random

# Directories
src_dir   = "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/data/training_images"
train_dir = "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/data/my_training"
val_dir   = "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/data/my_validation"
test_dir  = "/Users/zignov/Desktop/FRI/SecondYear/ComputerVision/data/my_testing"

# Ratios
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# List of filenames without extension
filenames = [os.path.splitext(f)[0] for f in os.listdir(src_dir) if f.endswith('.jpg') or f.endswith('.png')]
random.shuffle(filenames)

# Calculate split indexes
train_idx = int(len(filenames) * train_ratio)
val_idx = int(len(filenames) * (train_ratio + val_ratio))

# Split filenames
train_files = filenames[:train_idx]
val_files = filenames[train_idx:val_idx]
test_files = filenames[val_idx:]

# Function to move files
def move_files(file_list, dest_dir):
    for fname in file_list:
        shutil.move(f"{src_dir}/{fname}.jpg", f"{dest_dir}/{fname}.jpg")
        shutil.move(f"{src_dir}/{fname}.txt", f"{dest_dir}/{fname}.txt")

# Move files to respective directories
move_files(train_files, train_dir)
move_files(val_files, val_dir)
move_files(test_files, test_dir)
